In [1]:
import pandas as pd
import requests
import time
import random
from datetime import datetime, timedelta
 

In [2]:
INPUT_CSV = "NER_landslide_final_ML.csv"   # your existing positive dataset
OUTPUT_NEGATIVE_CSV = "NER_negative_examples.csv"
OUTPUT_COMBINED_CSV = "NER_landslide_combined_final.csv"
 

In [3]:
NEGATIVES_PER_LOCATION = 3      # ~3 negatives per positive -> good balance
EXCLUSION_WINDOW_DAYS = 15      # don't pick a "safe" date within this many days of a real event
MONSOON_MONTHS = [6, 7, 8, 9]   # June-September, when most landslides occur
YEARS_TO_SAMPLE = list(range(2010, 2024))

In [4]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

In [8]:
def load_positive_data():
    df = pd.read_csv(INPUT_CSV)
    df["event_date"] = pd.to_datetime(df["event_date"])
    df["landslide_occurred"] = 1
    return df
 
 
def get_unique_locations(df):
    """
    One row per unique (lat, lon) location, keeping the location-based
    attributes (elevation, slope, soil, admin info) that we'll reuse
    for negative examples at that same spot.
    """
    location_cols = [
        "latitude", "longitude", "elevation_m", "slope_deg",
        "FAOSOIL", "DOMSOI", "admin_division_name",
        "country_name", "country_code", "location_accuracy_km",
    ]
    location_cols = [c for c in location_cols if c in df.columns]
    unique_locs = df.drop_duplicates(subset=["latitude", "longitude"])[location_cols]
    return unique_locs.reset_index(drop=True)
 
 
# ---------------------------------------------------------
# STEP 2: Pick "safe" candidate dates for a location
# ---------------------------------------------------------
 
def get_known_event_dates(df, lat, lon):
    """All real event dates recorded at this exact location."""
    matches = df[(df["latitude"] == lat) & (df["longitude"] == lon)]
    return list(matches["event_date"])
 
 
def is_far_from_known_events(candidate_date, known_dates, window_days=EXCLUSION_WINDOW_DAYS):
    for known_date in known_dates:
        if abs((candidate_date - known_date).days) <= window_days:
            return False
    return True
 
 
def generate_safe_dates(lat, lon, known_dates, n_dates=NEGATIVES_PER_LOCATION):
    """
    Randomly picks n_dates monsoon-season dates that are NOT close to
    any known real landslide event at this location.
    """
    safe_dates = []
    attempts = 0
    max_attempts = n_dates * 20  # avoid infinite loop if a location is very event-dense
 
    while len(safe_dates) < n_dates and attempts < max_attempts:
        attempts += 1
        year = random.choice(YEARS_TO_SAMPLE)
        month = random.choice(MONSOON_MONTHS)
        day = random.randint(1, 28)  # 28 keeps it safe across all months
        candidate = datetime(year, month, day)
 
        if is_far_from_known_events(candidate, known_dates) and candidate not in safe_dates:
            safe_dates.append(candidate)
 
    return safe_dates
 
 
# ---------------------------------------------------------
# STEP 3: Fetch historical rainfall for a given lat/lon/date
# ---------------------------------------------------------
 
def fetch_historical_rainfall(lat, lon, target_date, retries=3, pause=1.0):
    """
    Pulls daily precipitation for the 14 days ending on target_date,
    then computes the same rolling features used in the positive dataset:
    rainfall_1d, rainfall_3d, rainfall_7d, rainfall_14d, max_rainfall_7d
    """
    start_date = target_date - timedelta(days=13)  # 14-day window inclusive of target_date
    end_date = target_date
 
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata",
    }
 
    for attempt in range(retries):
        try:
            resp = requests.get(ARCHIVE_URL, params=params, timeout=20)
            resp.raise_for_status()
            data = resp.json()
            daily_vals = data["daily"]["precipitation_sum"]  # 14 values, oldest -> newest
 
            # daily_vals[-1] is target_date itself
            rainfall_1d = daily_vals[-1]
            rainfall_3d = sum(daily_vals[-3:])
            rainfall_7d = sum(daily_vals[-7:])
            rainfall_14d = sum(daily_vals[-14:])
            max_rainfall_7d = max(daily_vals[-7:])
 
            return {
                "rainfall_1d": rainfall_1d,
                "rainfall_3d": rainfall_3d,
                "rainfall_7d": rainfall_7d,
                "rainfall_14d": rainfall_14d,
                "max_rainfall_7d": max_rainfall_7d,
            }
 
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(pause * (attempt + 1))
                continue
            print(f"Rainfall fetch failed for ({lat}, {lon}) on {target_date.date()}: {e}")
            return None
 
 
# ---------------------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------------------
 
def build_negative_dataset():
    print("Loading positive data...")
    df_pos = load_positive_data()
    unique_locs = get_unique_locations(df_pos)
    print(f"  -> {len(unique_locs)} unique locations found")
 
    negative_rows = []
 
    for i, loc in unique_locs.iterrows():
        lat, lon = loc["latitude"], loc["longitude"]
        known_dates = get_known_event_dates(df_pos, lat, lon)
 
        safe_dates = generate_safe_dates(lat, lon, known_dates)
 
        for safe_date in safe_dates:
            rainfall_data = fetch_historical_rainfall(lat, lon, safe_date)
            time.sleep(1.0)  # be polite to the free API
 
            if rainfall_data is None:
                continue  # skip if API failed even after retries
 
            row = loc.to_dict()  # copies elevation, slope, soil, admin info
            row.update(rainfall_data)
            row["event_date"] = safe_date
            row["landslide_occurred"] = 0
            row["landslide_trigger"] = "none"
            row["event_title"] = "Synthetic negative example"
            row["event_description"] = (
                f"No recorded landslide at this location on {safe_date.date()}"
            )
            negative_rows.append(row)
 
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(unique_locs)} locations "
                  f"({len(negative_rows)} negatives generated so far)")
 
    df_neg = pd.DataFrame(negative_rows)
    df_neg.to_csv(OUTPUT_NEGATIVE_CSV, index=False)
    print(f"\nSaved {len(df_neg)} negative examples to {OUTPUT_NEGATIVE_CSV}")
 
    return df_pos, df_neg
 
 
def combine_datasets(df_pos, df_neg):
    """
    Combines positive + negative into one final training dataset.
    Only keeps columns common to both, so the model sees a clean,
    consistent feature set.
    """
    common_cols = [c for c in df_pos.columns if c in df_neg.columns]
    df_final = pd.concat(
        [df_pos[common_cols], df_neg[common_cols]],
        ignore_index=True,
    )
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows
    df_final.to_csv(OUTPUT_COMBINED_CSV, index=False)
    print(f"\nFinal combined dataset: {len(df_final)} rows")
    print(df_final["landslide_occurred"].value_counts())
    print(f"Saved to {OUTPUT_COMBINED_CSV}")
    return df_final
 
 

In [9]:
df_pos, df_neg = build_negative_dataset()
df_final = combine_datasets(df_pos, df_neg)

Loading positive data...
  -> 349 unique locations found
  Processed 10/349 locations (30 negatives generated so far)
  Processed 20/349 locations (60 negatives generated so far)
  Processed 30/349 locations (90 negatives generated so far)
  Processed 40/349 locations (120 negatives generated so far)
  Processed 50/349 locations (150 negatives generated so far)
  Processed 60/349 locations (180 negatives generated so far)
  Processed 70/349 locations (210 negatives generated so far)
  Processed 80/349 locations (240 negatives generated so far)
  Processed 90/349 locations (270 negatives generated so far)
  Processed 100/349 locations (300 negatives generated so far)
  Processed 110/349 locations (330 negatives generated so far)
  Processed 120/349 locations (360 negatives generated so far)
  Processed 130/349 locations (390 negatives generated so far)
  Processed 140/349 locations (420 negatives generated so far)
  Processed 150/349 locations (450 negatives generated so far)
  Processe